In [30]:
import os 
import sys 
import ipynbname
import fnmatch
from pathlib import Path

import logging
from dotenv import load_dotenv
load_dotenv()

python_dir=os.getenv('python_dir')

if python_dir not in sys.path: 
    print(f"adding {python_dir} to sys path")
    sys.path.append(python_dir)

from common_utils import setup_logger
log_dir=os.getenv('log_dir')
log_file=os.path.join(log_dir, f"{ipynbname.name()}.log")
setup_logger('INFO',log_file)

logging.info(f"start  {ipynbname.name()}")

In [31]:
from pathlib import Path 
tmp_dir=os.getenv('tmp_dir')
workspace_dir=os.path.join(tmp_dir, "aimpro",f"{os.getpid()}")
Path(workspace_dir).mkdir(parents=True,exist_ok=True)
workspace_dir
#workspace_dir="/Users/feng/workspace/runtime/tmp/aimpro/29403"

'/tmp/aimpro/1613'

In [32]:
video_file_ext="MP4"
extract_fps=3
extract_img_fmt="jpg"
detection_chunk_size=60
pip=True

In [ ]:
# import pyperclip
# import os 
# raw_video_dir  = os.getenv('raw_video_dir')
# left_video_dir=os.path.join(raw_video_dir, 'left')
# right_video_dir=os.path.join(raw_video_dir, 'right')

# def file_sort_key(f):
#     name=os.path.basename(f)
#     name=name.split('.')[0].split('_')[-1]
#     return int(name)

# def gen_concat_cmd(video_dir, concat_file_name): 
#     files=fnmatch.filter(os.listdir(video_dir), f"*.{video_file_ext}")
#     files=[os.path.join(video_dir, f) for f in files ]
#     files=sorted(files, key=file_sort_key)
#     list_file_name=os.path.join(workspace_dir, f"{concat_file_name}.list.txt")
#     with open(list_file_name, 'w') as h : 
#         for f in files:
#             h.write(f"file '{f}' \n")
#     txt=  f"ffmpeg -f concat -safe 0 -i {list_file_name} -c copy {os.path.join(workspace_dir, concat_file_name)}"
#     pyperclip.copy(txt)
#     return txt 


In [28]:
# cmd=gen_concat_cmd(left_video_dir, 'left_all_in_one.MP4')
# !{cmd}

In [29]:
# cmd =gen_concat_cmd(right_video_dir, 'right_all_in_one.MP4')
# !{cmd}

In [34]:
left_videos=fnmatch.filter(os.listdir(left_video_dir), f"*.MOV")
right_videos=fnmatch.filter(os.listdir(right_video_dir), f"*.{video_file_ext}")
left_videos=sorted(left_videos, key=file_sort_key)
right_videos=sorted(right_videos, key=file_sort_key)

left_videos , right_videos

(['IMG_4844.MOV', 'IMG_4845.MOV', 'IMG_4846.MOV'],
 ['20260619205546_000024.MP4',
  '20260619210856_000025.MP4',
  '20260619212244_000026.MP4'])

In [35]:

# left_video_file=os.path.join(raw_video_dir, 'left','20260516181012_000004.MP4')
# right_video_file=os.path.join(raw_video_dir, 'right','r_20260516181403_000006.MP4')
# left_video_file=os.path.join(workspace_dir, 'left_all_in_one.MP4')
# right_video_file=os.path.join(workspace_dir, 'right_all_in_one.MP4')
quater=1
left_video_file=os.path.join(left_video_dir, left_videos[quater-1])
right_video_file=os.path.join(right_video_dir, right_videos[quater-1])
# left_video_file=os.path.join(workspace_dir, "left_all_in_one.MP4")
# right_video_file=os.path.join(workspace_dir, "right_all_in_one.MP4")
left_video_file, right_video_file

('/mnt/c/data/temp/aimpro_raw_video/left/IMG_4844.MOV',
 '/mnt/c/data/temp/aimpro_raw_video/right/20260619205546_000024.MP4')

In [36]:
from video.synchronizer import * 
raw_offset, meaning = find_offset_seconds(workspace_dir,left_video_file, right_video_file)
offset=round(raw_offset)
raw_offset, offset, meaning

(-1.998125, -2, 'video B started 1.998125 earlier than video A.')

In [37]:
from video.ffmpeg_wrapper import * 
left_video_duration= get_video_duration(left_video_file)
right_video_duration= get_video_duration(right_video_file)
left_video_duration,right_video_duration 

(709.176667, 709.933333)

In [38]:
left_frames_dir=os.path.join(workspace_dir, 'left')
Path(left_frames_dir).mkdir(parents=True, exist_ok=True)

right_frames_dir=os.path.join(workspace_dir, 'right')
Path(right_frames_dir).mkdir(parents=True, exist_ok=True)

In [39]:
def get_start_end_time(offset, left_duration, right_duration): 
    if offset <0: 
        # right start early
        right_start=-1*offset
        new_right_duration =  right_duration+offset 
        min_duration =min(left_duration, new_right_duration)
        return 0, min_duration, right_start, right_start+min_duration

    elif offset>0: 
        left_start=offset 
        new_left_duration = left_duration-offset
        min_duration=min(new_left_duration, right_duration)
        return left_start, left_start+ min_duration, 0, min_duration
ts= get_start_end_time(offset, left_video_duration, right_video_duration)
ts =[format_seconds_to_hhmmss(t) for t in ts]
left_start, left_end, right_start, right_end= ts[0],ts[1],ts[2],ts[3]
left_start, left_end, right_start, right_end

('00:00:00', '00:11:47.933', '00:00:02', '00:11:49.933')

In [40]:
file_name_pattern=f"%d.{extract_img_fmt}"
output = os.path.join(left_frames_dir, file_name_pattern)
cmd=f"ffmpeg -ss {left_start} -to {left_end} -i {left_video_file} -vf scale=1280:720,fps={extract_fps} {output}"
out =!{cmd}
print(out[:20])


['ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers', '  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)', '  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvp

In [41]:
output = os.path.join(right_frames_dir, file_name_pattern)
cmd=f"ffmpeg -ss {right_start} -to {right_end} -i {right_video_file} -vf scale=1280:720,fps={extract_fps} {output}"
out =!{cmd}
print(out[:1])


['ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers']


In [42]:
def rename_img_file_to_seconds_index(fd):
    files = fnmatch.filter(os.listdir(fd), f"*.{extract_img_fmt}")
    for f in files: 
        parts=f.split(".")
        file_index, ext=int(parts[0])-1, parts[1]
        second, index_in_second=divmod(file_index,extract_fps)
        new_file_name=f"{second}-{index_in_second}.{ext}"
        shutil.move(os.path.join(fd,f), os.path.join(fd,new_file_name))
rename_img_file_to_seconds_index(left_frames_dir)
rename_img_file_to_seconds_index(right_frames_dir)

In [43]:
from ml.detectors import * 
import pandas  as pd 

detection_chunk_size=240

def detect_basketball_and_players(video_frame_dir): 
    files=fnmatch.filter(os.listdir(video_frame_dir), f"*.{extract_img_fmt}")
    df=pd.DataFrame(data=files, columns=["file_name"])
    df['chunk']=range(df.shape[0])
    df['chunk']=df.chunk//detection_chunk_size
    def detect_chunk(gdf):
        logging.info(f"object detection chuck : {gdf.name}")
        files=[os.path.join(video_frame_dir,f) for f in gdf.file_name] 
        results=detect_objects_on_multiple_images(files)
        trios=[analyze_result(r) for r in results]
        num_of_basketballs, num_of_players , num_of_possessions, total_object_area= zip (*trios)
        gdf['has_baseketball']=num_of_basketballs
        gdf['has_baseketball']=gdf.has_baseketball>0
        gdf['has_possession']=num_of_possessions
        gdf['has_possession']=gdf.has_possession>0
        gdf['total_num_of_players']=num_of_players
        gdf['total_object_area']=total_object_area
        
        return gdf 
    ret= df.groupby('chunk').apply(detect_chunk)
    ret.sort_values(by="file_name", inplace=True)
    ret.to_csv(os.path.join(video_frame_dir, "frame_information.csv"), index=False)
    # return pd.concat(dfs)
    return ret
detect_basketball_and_players(left_frames_dir).shape , detect_basketball_and_players(right_frames_dir).shape 


((2124, 5), (2124, 5))

In [51]:
def agg_frame_information_df(df):
    def extract_second_and_index(fn):
        parts = fn.split(".")[0].split("-") 
        return parts[0], parts[1]
    pairs=df.file_name.apply(extract_second_and_index)
    df['second'], df["index_within_second"]=zip (*pairs)
    df['second'], df["index_within_second"]=df.second.astype(int), df.index_within_second.astype(int)
    adf=df.groupby('second').agg(
        {
            "has_baseketball":'any',
            "total_num_of_players":'sum' ,
            "has_possession":'any', 
            "total_object_area":'sum'   
        } 
    ).reset_index()
    adf.sort_values(by='second', inplace=True)
    return adf 

left_frames_df=pd.read_csv(os.path.join(left_frames_dir, "frame_information.csv"), index_col=None)
left_frames_df=agg_frame_information_df(left_frames_df)

right_frames_df=pd.read_csv(os.path.join(right_frames_dir, "frame_information.csv"), index_col=None)
right_frames_df=agg_frame_information_df(right_frames_df)

left_frames_df.shape, right_frames_df.shape 


((708, 5), (708, 5))

In [45]:
left_frames_df.head(2)

,second,has_baseketball,total_num_of_players,has_possession,total_object_area
0,0,False,0,False,0.000000
1,1,True,4,False,33395.107422


In [ ]:
mdf=left_frames_df.merge(right_frames_df, how='inner', on='second', suffixes=("_l","_r"))
mdf.head(2) 

,second,has_baseketball_l,total_num_of_players_l,has_possession_l,total_object_area_l,has_baseketball_r,total_num_of_players_r,has_possession_r,total_object_area_r
0,0,False,0,False,0.000000,False,8,False,53716.728516
1,1,True,4,False,33395.107422,False,6,False,28876.483398


In [ ]:
def smooth_left_right(ac): 
    s=ac.copy()
    prev,nxt=s.shift(1),s.shift(-1)
    mask=(s!=prev) & (s!=nxt) & (prev==nxt)
    ret=s.copy()
    ret[mask]=prev[mask]

    #force setting the first to the same as second; 
    #the last to the same as the second last;
    if ret.size>2:
        ret.iloc[0]=ret.iloc[1]
        ret.iloc[-1]=ret.iloc[-2]

    return ret 

def decide_active_camera(r):
    # is_timeout_or_break=r['total_num_of_players_l']<=2 and r['total_num_of_players_r']<=2
    # if is_timeout_or_break: 
    #      return "skip"
    
    ball_on_left= r["has_possession_l"] or r["has_baseketball_l"]
    ball_on_right= r["has_possession_r"] or r["has_baseketball_r"]

    only_on_left=ball_on_left and not ball_on_right
    only_on_right=ball_on_right and not ball_on_left

    if only_on_left:
         return 'left'
    elif only_on_right: 
        return 'right'
    else: 
        if r['total_num_of_players_l']>r['total_num_of_players_r']:
                return 'left'
        elif r['total_num_of_players_l']<r['total_num_of_players_r']:
                return 'right'
        else:
            if r['total_object_area_l']>r['total_object_area_r']:
                    return 'left'
            elif r['total_object_area_l']<r['total_object_area_r']:
                    return 'right'
    return None

mdf.sort_values(by='second', ascending=True, inplace=True)

mdf['active_camera']=mdf.apply(decide_active_camera, axis=1)
mdf.ffill(inplace=True)

#double smooth 
mdf['active_camera']=smooth_left_right(mdf.active_camera)
mdf['active_camera']=smooth_left_right(mdf.active_camera)
mdf['active_camera']=smooth_left_right(mdf.active_camera)
mdf.to_csv(os.path.join(workspace_dir,'active_camera.csv'), index=False)
mdf.head(2)

,second,has_baseketball_l,total_num_of_players_l,has_possession_l,total_object_area_l,has_baseketball_r,total_num_of_players_r,has_possession_r,total_object_area_r,active_camera
0,0,False,0,False,0.000000,False,8,False,53716.728516,left
1,1,True,4,False,33395.107422,False,6,False,28876.483398,left


In [48]:
mdf=pd.read_csv(os.path.join(workspace_dir,'active_camera.csv'), index_col=None)

def gen_hhmmss(side, offset, second): 
    #if offset <0 , then right start early; otherwise left start early. 
    if offset<0: 
        if side =='right': 
            second =float(second) +offset*(-1)
    else: 
        if side=='left': 
            second =float(second) +offset
    return format_seconds_to_hhmmss(second) 

def gen_ffmpeg_extract_commands(mdf):
    def gen_cmd(start_r,last_r): 
        if start_r.active_camera=='left':
            start_hhmmss=gen_hhmmss("left", offset, start_r.second)
            to_hhmmss=gen_hhmmss("left", offset, last_r.second)
            video_file_name=left_video_file
        else:
            start_hhmmss=gen_hhmmss("right", offset, start_r.second)
            to_hhmmss=gen_hhmmss("right", offset, last_r.second)
            video_file_name=right_video_file
        cmd1 = f"ffmpeg -ss {start_hhmmss} -to {to_hhmmss} -i {video_file_name}  -c copy part_to_be_replaced"
        return cmd1
    
    start_r, last_r=None, None  
    cmds=[]
    for r in mdf.itertuples():
        if start_r is None: 
            start_r, last_r=r, r 
            continue
        else: 
            if r.active_camera!=last_r.active_camera: 
                #need swtich. 
                cmds.append(gen_cmd(start_r, last_r))
                start_r, last_r=r, r 
            else:
                last_r=r 

    if start_r and last_r: 
        cmds.append(gen_cmd(start_r,last_r))       
    return cmds 

def gen_ffmpeg_extract_commands_with_pip(mdf):
    def gen_cmd(start_r,last_r): 
        
        if start_r.active_camera=='left':
            start_hhmmss=gen_hhmmss("left", offset, start_r.second)
            to_hhmmss=gen_hhmmss("left", offset, last_r.second)
            video_file_name=left_video_file

            pip_start_hhmmss=gen_hhmmss("right", offset, start_r.second)
            pip_to_hhmmss=gen_hhmmss("right", offset, last_r.second)
            pip_video_file_name=right_video_file
        else:
            start_hhmmss=gen_hhmmss("right", offset, start_r.second)
            to_hhmmss=gen_hhmmss("right", offset, last_r.second)
            video_file_name=right_video_file
            
            pip_start_hhmmss=gen_hhmmss("left", offset, start_r.second)
            pip_to_hhmmss=gen_hhmmss("left", offset, last_r.second)
            pip_video_file_name=left_video_file
            
        cmd1 = f"ffmpeg -ss {start_hhmmss} -to {to_hhmmss} -i {video_file_name}  -c copy part_to_be_replaced"
        cmd2 = f"ffmpeg -ss {pip_start_hhmmss} -to {pip_to_hhmmss} -i {pip_video_file_name}  -c copy {start_r.active_camera}-part_to_be_replaced" 
        return (cmd1,cmd2)
    
    start_r, last_r=None, None  
    cmds=[]
    for r in mdf.itertuples():
        if start_r is None: 
            start_r, last_r=r, r 
            continue
        else: 
            if r.active_camera!=last_r.active_camera: 
                #need swtich. 
                cmds.append(gen_cmd(start_r, last_r))
                start_r, last_r=r, r 
            else:
                last_r=r 

    if start_r and last_r: 
        cmds.append(gen_cmd(start_r,last_r))       
    return cmds 

if pip: 
    extract_cmds = gen_ffmpeg_extract_commands_with_pip(mdf)
    c, c_pip =extract_cmds[0]
    print(c)
    print(c_pip)
else: 
    extract_cmds = gen_ffmpeg_extract_commands(mdf)
    print(extract_cmds[0])



ffmpeg -ss 00:00:00 -to 00:00:04 -i /mnt/c/data/temp/aimpro_raw_video/left/IMG_4844.MOV  -c copy part_to_be_replaced
ffmpeg -ss 00:00:02 -to 00:00:06 -i /mnt/c/data/temp/aimpro_raw_video/right/20260619205546_000024.MP4  -c copy left-part_to_be_replaced


In [49]:
import pyperclip
from itertools import chain
def post_process_extract_cmds(extract_cmds): 
    def rpl(index, cmd): 
        part=os.path.join(workspace_dir, f"part-{index}.MP4")
        return cmd.replace('part_to_be_replaced', part)
    extract_cmds= [rpl(i,c) for i, c in enumerate(extract_cmds) ]
    text="\n".join(extract_cmds)
    with open(os.path.join(workspace_dir,'extract_part.sh'), 'w') as h: 
        h.write(text)
    
    return extract_cmds

def post_process_extract_cmds_pip(extract_cmds): 
    def rpl(index, cmd_pair): 
        c,c_pip=cmd_pair
        part=os.path.join(workspace_dir, f"part-{index}.MP4")
        c = c.replace('part_to_be_replaced', part)

        #for pip a bit tricky. 
        active_camera_and_place_holder=c_pip.split(" ")[-1]
        active_camera=active_camera_and_place_holder.split("-")[0]
        part=os.path.join(workspace_dir, f"pip-{active_camera}-part-{index}.MP4")
        c_pip = c_pip.replace(active_camera_and_place_holder, part)
        return c, c_pip
    extract_cmds= [rpl(i,c) for i, c in enumerate(extract_cmds) ]
    extract_cmds = list(chain.from_iterable(extract_cmds))
    print(len(extract_cmds))
    text="\n".join(extract_cmds)
    with open(os.path.join(workspace_dir,'extract_part.sh'), 'w') as h: 
        h.write(text)
    
    return extract_cmds

if pip:
    process_extract_cmds = post_process_extract_cmds_pip(extract_cmds)
else: 
    process_extract_cmds=post_process_extract_cmds(extract_cmds)

! bash {workspace_dir}/extract_part.sh 

146
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --

In [ ]:
def part_sort_key(fn):
    return int(fn.split(".")[0].split("-")[-1])

def produce_pip(part, pip_part): 
    #get active camera first. 
    # pip-right-part-8.MP4
    active_camera=pip_part.split("-")[1]
    output_part = f"with_pip_{part}"
    
    # Top-Left Corneroverlay=10:10
    # Top-Right Corneroverlay=W-w-10:10
    # Bottom-Left Corneroverlay=10:H-h-10
    # Bottom-Right Corneroverlay=W-w-10:H-h-10
    
    #place the overlay on the top left, if active camera is right. 
    #position_directive="overlay=20:20"
    position_directive="overlay=W-w-10:H-h-10"
    # if active_camera=='left': 
    #     #then place the pip at the right top corner. 
    #     position_directive="overlay=W-w-20:20"

    part =os.path.join(workspace_dir, part)
    pip_part=os.path.join(workspace_dir, pip_part)
    output_file=os.path.join(workspace_dir,output_part)

    #get the active camera first.                            "[1:v]scale=iw/4:-1, pad=iw+20:ih+20:10:10:color=white[pip];[0:v][pip]overlay=20:20:ts_sync_mode=vfr" 
    #return  f'ffmpeg -i {part} -i {pip_part} -filter_complex "[1:v]scale=iw/4:-1[pip];[0:v][pip]{position_directive}:shortest=1" -c:a copy {output_file}'
    return  f'ffmpeg -i {part} -i {pip_part} -filter_complex "[1:v]scale=iw/4:-1,pad=iw+5:ih+5:2:2:color=greenyellow [pip];[0:v][pip]{position_directive}:shortest=1" -c:a copy {output_file}'
    

    

if pip: 
    parts=fnmatch.filter(os.listdir(workspace_dir),"part*.MP4")
    pipparts =fnmatch.filter(os.listdir(workspace_dir), "pip*part*.MP4")
    parts.sort(key=part_sort_key)
    pipparts.sort(key=part_sort_key)
    main_pip_pairs=list(zip(parts,pipparts))
    for (p, pip) in main_pip_pairs: 
        cmd= produce_pip(p, pip )
        !{cmd}


ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [53]:
if not pip: 
    def gen_concat_command(extract_cmds, output_file_name_path): 
        list_file_name=os.path.join(workspace_dir, 'part-list.txt')
        with open(list_file_name, 'w') as h: 
            for i in range(len(extract_cmds)):
                p=os.path.join(workspace_dir,f"part-{i}.MP4")  
                h.write(f"file '{p}' \n")
        txt=  f"ffmpeg -f concat -safe 0 -i {list_file_name} -c copy {output_file_name_path}"
        pyperclip.copy(txt)
        return txt 

    output_path_file=os.path.join(raw_video_dir,f'pip-test-{quater}.MP4')
    concat_cmd= gen_concat_command(process_extract_cmds, output_path_file)
    ! {concat_cmd}
else: 
    def gen_concat_command(output_file_name_path): 
        part_list=list(fnmatch.filter(os.listdir(workspace_dir), "with_pip_part*.MP4"))
        part_list.sort(key=part_sort_key)
        part_list=[os.path.join(workspace_dir, p) for p in part_list]
        list_file_name=os.path.join(workspace_dir, 'part-list.txt')
        with open(list_file_name, 'w') as h: 
            for p  in part_list:
                h.write(f"file '{p}' \n")

        txt=  f"ffmpeg -f concat -safe 0 -i {list_file_name} -c copy {output_file_name_path}"
        pyperclip.copy(txt)
        return txt 

    output_path_file=os.path.join(raw_video_dir,f'war-vs-knox-{quater}.MP4')
    concat_cmd= gen_concat_command(output_path_file)
    ! {concat_cmd}



ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [ ]:
! rm -rf {workspace_dir}

In [104]:
#compress. 
#!ffmpeg -i {output_path_file} -c:v libx265 -crf 28 -preset slow -c:a aac -tag:v hvc1 -b:a 128k 265-{output_path_file}
